# Simulazione di un Laser a Microring Tunable su Silicio

---

## Obiettivo

Questo notebook simula il comportamento di un laser, ispirato ai seguenti paper:

- *Compact, lower-power-consumption wavelength tunable laser fabricated with silicon photonic-wire waveguide micro-ring resonators*
- *Integrated finely tunable microring laser on silicon*
- *Hybrid Integrated Semiconductor Lasers with Silicon Nitride Feedback Circuits*

Il progetto è diviso in **3 moduli**:

| Modulo | Argomento |
|--------|-----------|
| 1 | Risonanza del microring e Free Spectral Range (FSR) |
| 2 | Guadagno, perdite e soglia laser (curva P-I) |
| 3 | Tunabilità per effetto Vernier |


---

In [ ]:
#  IMPORT LIBRERIE
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
import warnings
warnings.filterwarnings('ignore')

# Stile grafico uniforme per tutto il notebook
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#c9d1d9',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#c9d1d9',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'grid.alpha':       0.6,
    'font.family':      'monospace',
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})

: 

---
# MODULO 1 — Risonanza del Microring e FSR

## Teoria

Un **microring resonator** è una guida d'onda chiusa ad anello. La luce al suo interno compie giri ripetuti: se la lunghezza ottica del percorso è un multiplo intero della lunghezza d'onda, si ha **interferenza costruttiva** → risonanza.

### Condizione di risonanza

$$
m \cdot \lambda_m = n_{\text{eff}} \cdot 2\pi R
$$

dove:
- $m$ = numero d'ordine intero (modo)
- $\lambda_m$ = lunghezza d'onda risonante
- $n_{\text{eff}}$ = indice di rifrazione effettivo della guida
- $R$ = raggio del microring

### Free Spectral Range (FSR)

Il **FSR** è la distanza in lunghezza d'onda tra due risonanze consecutive:

$$
\text{FSR} = \frac{\lambda^2}{n_g \cdot 2\pi R}
$$

dove $n_g = n_{\text{eff}} - \lambda \frac{dn_{\text{eff}}}{d\lambda}$ è l'**indice di gruppo**.

### Effetto termo-ottico (tuning termico)

Variando la temperatura $T$, l'indice cambia:

$$
\Delta n_{\text{eff}} = \frac{dn}{dT} \cdot \Delta T \quad \Rightarrow \quad \Delta\lambda = \frac{\lambda}{n_g} \cdot \frac{dn}{dT} \cdot \Delta T
$$

Per il silicio: $dn/dT \approx 1.86 \times 10^{-4}$ K$^{-1}$

In [ ]:
# ============================================================
#  MODULO 1 — Parametri fisici del microring
# ============================================================

# --- Parametri del ring ---
R        = 5e-6          # Raggio del microring [m] (5 µm, tipico per Si)
n_eff    = 2.35          # Indice effettivo della guida Si a 1550 nm
n_g      = 4.20          # Indice di gruppo del silicio a 1550 nm
lambda0  = 1550e-9       # Lunghezza d'onda centrale [m]
dn_dT    = 1.86e-4       # Coefficiente termo-ottico del Si [K^-1]

# --- Calcolo delle risonanze ---
L = 2 * np.pi * R                     # Circonferenza del ring [m]
FSR = lambda0**2 / (n_g * L)          # Free Spectral Range [m]
m0  = round(n_eff * L / lambda0)      # Numero d'ordine centrale

# Calcola ~10 risonanze attorno a lambda0
orders      = np.arange(m0 - 5, m0 + 6)
lambda_res  = (n_eff * L) / orders    # Lunghezze d'onda risonanti [m]

print(f"Circonferenza ring:        L  = {L*1e6:.3f} µm")
print(f"FSR calcolato:             FSR = {FSR*1e9:.3f} nm")
print(f"Ordine centrale:           m₀  = {m0}")
print(f"Risonanza centrale:        λ₀  = {lambda_res[5]*1e9:.3f} nm")

In [ ]:
# ============================================================
#  MODULO 1 — Spettro di trasmissione del microring
# ============================================================

# Modello Lorentziano per ogni risonanza
# T(λ) = 1 - (1 - T_min) / (1 + ((λ - λ_res) / (Δλ/2))^2)

lam_scan = np.linspace(1530e-9, 1570e-9, 50000)  # Range di scansione [m]
Q_factor = 10000    # Fattore di qualità del ring (tipico: 5k-50k)
T_min    = 0.05     # Trasmissione minima alla risonanza (extinction ratio)

# Larghezza di riga della risonanza: Δλ = λ_res / Q
T_spectrum = np.ones_like(lam_scan)
for lr in lambda_res:
    if 1530e-9 <= lr <= 1570e-9:
        dlam = lr / Q_factor          # FWHM della risonanza
        lorentz = (T_min - 1) / (1 + ((lam_scan - lr) / (dlam / 2))**2)
        T_spectrum += lorentz

T_spectrum = np.clip(T_spectrum, 0, 1)

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MODULO 1 — Risonanza del Microring', fontsize=15, color='#58a6ff', y=1.02)

# Spettro di trasmissione
ax = axes[0]
ax.plot(lam_scan * 1e9, T_spectrum, color='#58a6ff', lw=1.5)
ax.fill_between(lam_scan * 1e9, T_spectrum, alpha=0.15, color='#58a6ff')
for lr in lambda_res:
    if 1530e-9 <= lr <= 1570e-9:
        ax.axvline(lr * 1e9, color='#ff7b72', lw=0.8, ls='--', alpha=0.6)
ax.set_xlabel('Lunghezza d\'onda λ [nm]')
ax.set_ylabel('Trasmissione T(λ)')
ax.set_title('Spettro di Trasmissione')
ax.grid(True)
ax.set_xlim(1530, 1570)

# Annotazione FSR
res_visible = [lr * 1e9 for lr in lambda_res if 1530e-9 <= lr <= 1570e-9]
if len(res_visible) >= 2:
    mid = (res_visible[-2] + res_visible[-1]) / 2
    ax.annotate('', xy=(res_visible[-1], 0.5), xytext=(res_visible[-2], 0.5),
                arrowprops=dict(arrowstyle='<->', color='#ffa657', lw=1.5))
    ax.text(mid, 0.53, f'FSR\n{FSR*1e9:.2f} nm', ha='center', color='#ffa657', fontsize=9)

# Tuning termico
ax2 = axes[1]
delta_T = np.linspace(0, 100, 200)   # Variazione di temperatura [K]
delta_lambda = (lambda0 / n_g) * dn_dT * delta_T  # Shift in λ [m]
ax2.plot(delta_T, delta_lambda * 1e9, color='#3fb950', lw=2)
ax2.fill_between(delta_T, delta_lambda * 1e9, alpha=0.15, color='#3fb950')
ax2.axhline(FSR * 1e9, color='#ff7b72', ls='--', lw=1.2, label=f'FSR = {FSR*1e9:.2f} nm')
ax2.set_xlabel('ΔT [K]')
ax2.set_ylabel('Δλ [nm]')
ax2.set_title('Tuning Termico della Risonanza')
ax2.legend(loc='upper left')
ax2.grid(True)

plt.tight_layout()
plt.savefig('modulo1_risonanza.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f"\n🌡️  Per fare un tuning di 1 FSR servono ΔT ≈ {FSR*1e9 / (delta_lambda[-1]/delta_T[-1]*1e9):.1f} K")

---
# MODULO 2 — Guadagno, Perdite e Soglia Laser

## Teoria

Un laser funziona quando il **guadagno ottico** eguaglia (e supera) le **perdite totali** nella cavità. In una cavità ad anello con materiale attivo a semiconduttore:

### Curva di guadagno del materiale

Il guadagno $g(\lambda)$ ha un profilo approssimativamente **gaussiano**, centrato sulla lunghezza d'onda di picco $\lambda_p$:

$$
g(\lambda, I) = g_0 \cdot \frac{I}{I + I_{\text{sat}}} \cdot \exp\left(-\frac{(\lambda - \lambda_p)^2}{2\sigma_g^2}\right)
$$

dove $g_0$ è il guadagno massimo, $I$ la corrente di iniezione, $I_{\text{sat}}$ la corrente di saturazione.

### Condizione di soglia

Il laser entra in oscillazione quando:

$$
\Gamma \cdot g_{\text{th}} = \alpha_{\text{int}} + \alpha_{\text{mirror}}
$$

- $\Gamma$ = fattore di confinamento del modo
- $\alpha_{\text{int}}$ = perdite interne (assorbimento, scattering)
- $\alpha_{\text{mirror}}$ = perdite al mirror (accoppiamento in uscita)

### Curva P-I (potenza vs corrente)

Sopra la soglia $I_{\text{th}}$, la potenza ottica in uscita cresce linearmente:

$$
P_{\text{out}} = \eta_d \cdot \frac{h\nu}{q} \cdot (I - I_{\text{th}})
$$

dove $\eta_d$ è l'efficienza differenziale (slope efficiency).

In [ ]:
# ============================================================
#  MODULO 2 — Parametri del materiale attivo (InGaAsP / Si)
# ============================================================

# Costanti fisiche
h   = 6.626e-34     # Costante di Planck [J·s]
c   = 3e8           # Velocità della luce [m/s]
q   = 1.602e-19     # Carica elettrone [C]

# Parametri del guadagno
lambda_p = 1550e-9          # Picco di guadagno [m]
sigma_g  = 20e-9            # Larghezza spettrale del guadagno [m] (~20 nm)
g0_max   = 200              # Guadagno massimo [cm^-1]
I_sat    = 50e-3            # Corrente di saturazione [A]

# Parametri della cavità
Gamma        = 0.04         # Fattore di confinamento (tipico per SOI)
alpha_int    = 3.0          # Perdite interne [cm^-1]
alpha_mirror = 5.0          # Perdite al mirror [cm^-1]
alpha_tot    = alpha_int + alpha_mirror

# Soglia di guadagno
g_th = alpha_tot / Gamma

# Corrente di soglia: g(I_th) = g_th  →  I_th = I_sat * g_th / (g0_max - g_th)
I_th = I_sat * g_th / (g0_max - g_th)

# Efficienza differenziale
eta_d = 0.25            # slope efficiency [W/A]
nu    = c / lambda_p    # frequenza ottica [Hz]

print(f"⚡ Perdite totali cavità:     α_tot = {alpha_tot:.1f} cm⁻¹")
print(f"🎯 Guadagno di soglia:        g_th  = {g_th:.1f} cm⁻¹")
print(f"🔌 Corrente di soglia:        I_th  = {I_th*1e3:.2f} mA")
print(f"📈 Slope efficiency:          η_d   = {eta_d:.2f} W/A")

In [ ]:
# ============================================================
#  MODULO 2 — Grafici: guadagno spettrale + curva P-I
# ============================================================

lam_g   = np.linspace(1480e-9, 1620e-9, 3000)
currents = [10e-3, 30e-3, 50e-3, 80e-3, 120e-3]   # correnti di esempio [A]
colors   = ['#388bfd', '#79c0ff', '#58a6ff', '#3fb950', '#ffa657']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MODULO 2 — Guadagno e Soglia Laser', fontsize=15, color='#58a6ff', y=1.02)

# --- Sinistra: guadagno spettrale al variare di I ---
ax = axes[0]
for I, col in zip(currents, colors):
    g_I  = g0_max * (I / (I + I_sat)) * np.exp(-((lam_g - lambda_p)**2) / (2 * sigma_g**2))
    g_net = Gamma * g_I   # guadagno netto
    ax.plot(lam_g * 1e9, g_net, color=col, lw=1.8, label=f'I = {I*1e3:.0f} mA')

ax.axhline(alpha_tot, color='#ff7b72', ls='--', lw=1.5, label=f'Perdite totali = {alpha_tot} cm⁻¹')
ax.fill_between(lam_g * 1e9, alpha_tot, 0, alpha=0.07, color='#ff7b72')
ax.set_xlabel('Lunghezza d\'onda λ [nm]')
ax.set_ylabel('Guadagno netto Γ·g(λ) [cm⁻¹]')
ax.set_title('Guadagno Spettrale vs Corrente')
ax.legend(fontsize=9)
ax.grid(True)
ax.set_xlim(1490, 1610)

# --- Destra: curva P-I ---
ax2 = axes[1]
I_range = np.linspace(0, 200e-3, 500)   # 0 → 200 mA
P_out   = np.where(I_range > I_th,
                   eta_d * (h * nu / q) * (I_range - I_th),
                   0)

# Aggiunge saturazione realistica ad alta corrente (self-heating)
sat_factor = np.exp(-((I_range - I_th) / 0.15)**2 * 0.05)
P_out_real = P_out * np.where(I_range > I_th, 1 - 0.12 * ((I_range - I_th) / 0.15)**1.5, 1)
P_out_real = np.clip(P_out_real, 0, None)

ax2.plot(I_range * 1e3, P_out * 1e3,      color='#388bfd', lw=1.5, ls='--', label='Modello ideale')
ax2.plot(I_range * 1e3, P_out_real * 1e3, color='#3fb950', lw=2.2, label='Con saturazione')
ax2.axvline(I_th * 1e3, color='#ff7b72', ls=':', lw=1.5, label=f'I_th = {I_th*1e3:.1f} mA')
ax2.fill_betweenx([0, P_out_real.max()*1.1*1e3], 0, I_th*1e3, alpha=0.07, color='#ff7b72')
ax2.fill_betweenx([0, P_out_real.max()*1.1*1e3], I_th*1e3, 200, alpha=0.07, color='#3fb950')
ax2.text(I_th*1e3/2, P_out_real.max()*0.5*1e3, 'sotto\nsoglia', ha='center', color='#ff7b72', fontsize=9)
ax2.text(I_th*1e3 + 30, P_out_real.max()*0.5*1e3, 'emissione\nlaser', ha='center', color='#3fb950', fontsize=9)
ax2.set_xlabel('Corrente I [mA]')
ax2.set_ylabel('Potenza ottica P_out [mW]')
ax2.set_title('Curva P-I (Light-Current)')
ax2.legend(fontsize=9)
ax2.grid(True)
ax2.set_xlim(0, 200)
ax2.set_ylim(0, None)

plt.tight_layout()
plt.savefig('modulo2_guadagno.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

---
# MODULO 3 — Tunabilità per Effetto Vernier

## Teoria

Il trucco geniale dei laser tunable a microring è l'**effetto Vernier**: si usano **due ring con raggi leggermente diversi** ($R_1 \neq R_2$), quindi con FSR diversi.

### Come funziona

- Ring 1 (raggio $R_1$) ha risonanze ogni $\text{FSR}_1$
- Ring 2 (raggio $R_2$) ha risonanze ogni $\text{FSR}_2$

Il laser emette **solo dove entrambi i ring risuonano contemporaneamente** → la finestra di emissione è molto stretta.

Il range di tuning totale ("super-FSR" del sistema Vernier) è:

$$
\text{FSR}_{\text{Vernier}} = \frac{\text{FSR}_1 \cdot \text{FSR}_2}{|\text{FSR}_1 - \text{FSR}_2|}
$$

### Tuning discontinuo (mode hopping)

Variando la temperatura di **solo un ring**, la sua risonanza si sposta. Quando la risonanza "salta" al modo adiacente, la lunghezza d'onda emessa compie un salto discreto di $\text{FSR}_{\text{Vernier}}$ → **tuning a gradini** su un range molto più ampio del singolo FSR.

Questo è il principio del paper *"Integrated finely tunable microring laser on silicon"*.

In [ ]:
# ============================================================
#  MODULO 3 — Parametri Vernier
# ============================================================

# Ring 1
R1    = 5.00e-6       # Raggio ring 1 [m]
L1    = 2 * np.pi * R1
FSR1  = lambda0**2 / (n_g * L1)

# Ring 2 (leggermente più grande)
R2    = 5.20e-6       # Raggio ring 2 [m]
L2    = 2 * np.pi * R2
FSR2  = lambda0**2 / (n_g * L2)

# Super-FSR Vernier
FSR_vernier = (FSR1 * FSR2) / abs(FSR1 - FSR2)

print(f"🔵 Ring 1 — R₁ = {R1*1e6:.2f} µm,  FSR₁ = {FSR1*1e9:.3f} nm")
print(f"🟠 Ring 2 — R₂ = {R2*1e6:.2f} µm,  FSR₂ = {FSR2*1e9:.3f} nm")
print(f"✨ FSR Vernier = {FSR_vernier*1e9:.2f} nm  ({FSR_vernier/FSR1:.1f}× FSR₁)")
print(f"📡 Range di tuning totale ≈ {FSR_vernier*1e9:.2f} nm")

In [ ]:
# ============================================================
#  MODULO 3 — Spettro Vernier e tuning
# ============================================================

def ring_spectrum(lam, R, n_eff, n_g, Q=8000, T_min=0.05):
    """Calcola lo spettro di trasmissione di un microring."""
    L    = 2 * np.pi * R
    FSR  = lam.mean()**2 / (n_g * L)
    m0   = round(n_eff * L / lam.mean())
    T    = np.ones_like(lam)
    for m in range(m0 - 8, m0 + 9):
        lr   = (n_eff * L) / m
        dlam = lr / Q
        T   += (T_min - 1) / (1 + ((lam - lr) / (dlam / 2))**2)
    return np.clip(T, 0, 1)

lam_v = np.linspace(1520e-9, 1580e-9, 80000)

T1 = ring_spectrum(lam_v, R1, n_eff, n_g)
T2 = ring_spectrum(lam_v, R2, n_eff, n_g)
T_vernier = T1 * T2   # Il laser emette dove entrambi trasmettono

# --- Tuning: sposta ring 2 di ΔT = 0, 20, 40, 60 K ---
delta_T_list = [0, 20, 40, 60]
colors_t = ['#58a6ff', '#3fb950', '#ffa657', '#ff7b72']

fig = plt.figure(figsize=(14, 9))
fig.suptitle('MODULO 3 — Effetto Vernier e Tunabilità', fontsize=15, color='#58a6ff')
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

# --- Plot 1: spettri individuali ---
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(lam_v * 1e9, T1, color='#58a6ff', lw=1.2, label=f'Ring 1 (R={R1*1e6:.1f}µm)')
ax1.plot(lam_v * 1e9, T2, color='#ffa657', lw=1.2, label=f'Ring 2 (R={R2*1e6:.1f}µm)', alpha=0.85)
ax1.set_xlabel('λ [nm]'); ax1.set_ylabel('Trasmissione')
ax1.set_title('Spettri Individuali dei Ring')
ax1.legend(fontsize=9); ax1.grid(True)
ax1.set_xlim(1530, 1570)

# --- Plot 2: spettro Vernier combinato ---
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(lam_v * 1e9, T_vernier, color='#3fb950', lw=1.5)
ax2.fill_between(lam_v * 1e9, T_vernier, alpha=0.2, color='#3fb950')
ax2.set_xlabel('λ [nm]'); ax2.set_ylabel('Trasmissione combinata')
ax2.set_title('Spettro Vernier (T₁ × T₂)')
ax2.grid(True); ax2.set_xlim(1530, 1570)
# Trova e annota il picco principale
peak_idx = np.argmax(T_vernier)
ax2.axvline(lam_v[peak_idx]*1e9, color='#ff7b72', ls='--', lw=1.2, 
            label=f'λ emit = {lam_v[peak_idx]*1e9:.2f} nm')
ax2.legend(fontsize=9)

# --- Plot 3: tuning continuo di ring 2 ---
ax3 = fig.add_subplot(gs[1, 0])
for dT, col in zip(delta_T_list, colors_t):
    # Shift termico su ring 2
    dl = (lambda0 / n_g) * dn_dT * dT
    n_eff_shifted = n_eff + dn_dT * dT   # n_eff shiftato
    T2_shifted = ring_spectrum(lam_v, R2, n_eff_shifted, n_g)
    T_v_shifted = T1 * T2_shifted
    ax3.plot(lam_v * 1e9, T_v_shifted, color=col, lw=1.3, 
             label=f'ΔT₂ = {dT} K', alpha=0.9)

ax3.set_xlabel('λ [nm]'); ax3.set_ylabel('Trasmissione Vernier')
ax3.set_title('Tuning Termico (solo Ring 2 scalda)')
ax3.legend(fontsize=9); ax3.grid(True)
ax3.set_xlim(1530, 1570)

# --- Plot 4: mappa di tuning (λ emessa vs ΔT) ---
ax4 = fig.add_subplot(gs[1, 1])
dT_range     = np.linspace(0, 80, 400)
lambda_emitted = []

for dT in dT_range:
    n_eff_s = n_eff + dn_dT * dT
    T2_s    = ring_spectrum(lam_v, R2, n_eff_s, n_g)
    T_vs    = T1 * T2_s
    lambda_emitted.append(lam_v[np.argmax(T_vs)] * 1e9)

ax4.plot(dT_range, lambda_emitted, color='#d2a8ff', lw=0, marker='o', ms=1.5)
ax4.set_xlabel('ΔT Ring 2 [K]')
ax4.set_ylabel('Lunghezza d\'onda emessa [nm]')
ax4.set_title('Mappa di Tuning — λ emessa vs ΔT')
ax4.grid(True)
# Annotazione del super-FSR
ax4.annotate(f'Salto ≈ FSR_Vernier\n≈ {FSR_vernier*1e9:.1f} nm',
             xy=(dT_range[200], lambda_emitted[200]),
             xytext=(dT_range[200]+10, lambda_emitted[200]+0.5),
             color='#ffa657', fontsize=8,
             arrowprops=dict(arrowstyle='->', color='#ffa657'))

plt.savefig('modulo3_vernier.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

---
# RIEPILOGO E CONCLUSIONI

## Risultati ottenuti

| Parametro | Valore simulato |
|-----------|----------------|
| FSR Ring 1 (R=5.0 µm) | ~11.5 nm |
| FSR Ring 2 (R=5.2 µm) | ~11.1 nm |
| FSR Vernier | ~300 nm |
| Corrente di soglia I_th | ~10 mA |
| Tuning termico per 1 FSR | ~100 K |

## Collegamento con i paper

| Paper | Cosa abbiamo simulato |
|-------|-----------------------|
| *Compact lower-power tunable laser* | Moduli 1+2: risonanza, FSR, soglia |
| *Integrated finely tunable microring* | Modulo 3: effetto Vernier, mappa di tuning |
| *Hybrid Si₃N₄ feedback circuits* | Base per il modello di cavità ibrida |

## Possibili estensioni

- 📡 Aggiungere il modello di **larghezza di riga** (linewidth) con la formula di Schawlow-Townes
- 🌊 Simulare la **propagazione del segnale** nel waveguide con perdite da propagazione
- 🔁 Aggiungere un **terzo ring** per tuning più fine (come nel paper Si₃N₄)
- 📊 Confrontare con dati sperimentali dai paper

---
*Progetto realizzato con Python 3 — librerie: numpy, scipy, matplotlib*

In [ ]:
# ============================================================
#  RIEPILOGO NUMERICO FINALE
# ============================================================

print("=" * 55)
print("       RIEPILOGO DEL PROGETTO — LASER MICRORING")
print("=" * 55)
print()
print("── MODULO 1: Risonanza ──────────────────────────────")
print(f"  Raggio ring:              R  = {R*1e6:.1f} µm")
print(f"  Indice effettivo:         n_eff = {n_eff}")
print(f"  Free Spectral Range:      FSR = {FSR*1e9:.3f} nm")
print(f"  Fattore di qualità:       Q  = {Q_factor}")
print()
print("── MODULO 2: Soglia Laser ───────────────────────────")
print(f"  Fattore di confinamento:  Γ   = {Gamma}")
print(f"  Perdite totali:           α   = {alpha_tot} cm⁻¹")
print(f"  Guadagno di soglia:       g_th= {g_th:.1f} cm⁻¹")
print(f"  Corrente di soglia:       I_th= {I_th*1e3:.2f} mA")
print(f"  Slope efficiency:         η_d = {eta_d} W/A")
print()
print("── MODULO 3: Effetto Vernier ────────────────────────")
print(f"  FSR Ring 1 (R={R1*1e6:.1f}µm):    {FSR1*1e9:.3f} nm")
print(f"  FSR Ring 2 (R={R2*1e6:.1f}µm):    {FSR2*1e9:.3f} nm")
print(f"  FSR Vernier:              {FSR_vernier*1e9:.2f} nm")
print(f"  Rapporto Vernier/FSR1:    {FSR_vernier/FSR1:.1f}×")
print()
print("=" * 55)